# 03_01 — TabEBM per-member (01 corner 와 bit-exact 일치)

**목적**: 01 에서 만든 K 개 ensemble member 와 **완전히 같은 4-corner 세트** 위에서  
TabEBM 표준 SGLD 를 K 번 돌려 K 개 sample set 생성.

- 01 의 corner_seed = `split_seed + k` (k = 0..K-1) 와 같은 숫자를 TabEBM.generate 의 seed 에 전달
- `ensemble_methods.build_surrogate_data` 가 MT19937 (RandomState) 로 바뀌었기 때문에 corner **set 단위 일치**
- 결과 파일: `samples/split_{i}/tabebm_m{k}/c{c}.npy`

04 에서 비교:
- `tabebm_m0` = 01 멤버 0 corner 위 단일 TabEBM
- `mean(tabebm_m0..m{K-1})` = K 멤버 평균 (pandas `.mean(axis=1)`)
- `vp_*` = 같은 K 멤버의 VP-SGLD aggregation

→ **같은 corner 풀 위에서 aggregation 방식 (단일 vs K-mean vs VP μ̂Σ̂) 차이** 공정 측정.

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as _mp
try: _mp.set_start_method('spawn', force=True)
except RuntimeError: pass

os.chdir('/home/work/JooKyung/TabEBM')
sys.path.insert(0, 'experiments'); sys.path.insert(0, 'src')

import numpy as np

def _fmt(s):
    s = int(s); m, s = divmod(s, 60)
    return f'{m}m{s:02d}s' if m else f'{s}s'

print('ready')

ready


## 1. EVAL_DIR 선택 (01 에서 생성)

In [2]:
fair_root = Path('experiments/fair_eval')
if fair_root.exists():
    for p in sorted(fair_root.iterdir()):
        if p.is_dir() and (p / 'config.json').exists():
            cfg = json.loads((p / 'config.json').read_text())
            ens_ok = (p / 'ensembles' / 'split_0' / 'c0' / 'meta.json').exists()
            smp_ok = (p / 'samples').exists()
            print(f'  {p.name}  K={cfg["K"]} methods={cfg["methods"]}  '
                  f'ens={"OK" if ens_ok else "--"}  samples={"OK" if smp_ok else "--"}')
else:
    print('  (없음 — 01 먼저 실행)')

  20260417_215937  K=10 methods=['Distance']  ens=OK  samples=OK
  20260418_221446  K=10 methods=['Distance']  ens=OK  samples=--
  20260419_081846  K=10 methods=['Distance']  ens=OK  samples=--
  20260419_171832_biodeg_n100_K10  K=10 methods=['Distance']  ens=OK  samples=--


In [ ]:
_candidates = sorted([
    p for p in fair_root.iterdir()
    if p.is_dir() and (p / 'config.json').exists()
       and (p / 'ensembles' / 'split_0' / 'c0' / 'meta.json').exists()
])
EVAL_DIR = _candidates[-1] if _candidates else Path('experiments/fair_eval/NONE')
# EVAL_DIR = Path('experiments/fair_eval/20260417_215937')   # <-- legacy stock (재현 불가)
assert (EVAL_DIR / 'config.json').exists(), f'없음: {EVAL_DIR}'

config = json.loads((EVAL_DIR / 'config.json').read_text())
K = config['K']
classes = config['classes']
print(f'EVAL_DIR: {EVAL_DIR}')
print(f'  K={K}, methods={config["methods"]}, n_splits={config["n_splits"]}')
print(f'  dataset={config["dataset"]}, n_real={config["n_real"]}, classes={classes}')

# 01 의 Distance 설정에서 distance_negative_class 추출
method_params = config.get('method_params', {})
dist_cfg = method_params.get('Distance', {})
if dist_cfg.get('mode') == 'fixed':
    DIST_NEG = float(dist_cfg['value'])
    print(f'  distance_negative_class = {DIST_NEG} (01 config 에서 자동 추출)')
else:
    raise ValueError(
        f'01 이 Distance fixed 모드가 아님 (mode={dist_cfg.get("mode")}). '
        f'per-member 비교는 고정 distance 에서만 의미 있음.')

## 2. TabEBM SGLD 하이퍼파라미터

K 멤버 모두 **같은 SGLD 파라미터**. seed 만 k 마다 다름 (01 의 corner_seed 와 일치시키기 위함).

In [ ]:
# === TabEBM 기본 SGLD 하이퍼파라미터 (논문 Table 1 기본값) ===
SGLD_STEPS      = 200              # 논문 기본값
SGLD_STEP_SIZE  = 0.1              # 논문 기본값
SGLD_NOISE_STD  = 0.01             # 논문 기본값
STARTING_NOISE  = 0.01             # 논문 기본값

print(f'SGLD: steps={SGLD_STEPS}, step_size={SGLD_STEP_SIZE}, '
      f'noise={SGLD_NOISE_STD}, start_noise={STARTING_NOISE}')
print(f'distance_negative_class = {DIST_NEG} (01 과 동일)')
print(f'K = {K} members per split  →  seed = split_seed + k  (01 의 corner_seed 와 set-level 일치)')

## 3. 멤버 config 조립

K 개 멤버 각각 `tabebm_m{k}` 로 명명. 모두 같은 SGLD 설정 + 다른 seed_offset = k.

In [ ]:
TABEBM_CONFIGS = [
    dict(
        name=f'tabebm_m{k}',
        sgld_steps=SGLD_STEPS,
        sgld_step_size=SGLD_STEP_SIZE,
        sgld_noise_std=SGLD_NOISE_STD,
        starting_point_noise_std=STARTING_NOISE,
        distance_negative_class=DIST_NEG,
        seed_offset=k,                  # worker: seed=split_seed + k (= 01 의 corner_seed)
    )
    for k in range(K)
]

print(f'{len(TABEBM_CONFIGS)} member configs:')
for c in TABEBM_CONFIGS[:3]:
    print(f'  {c["name"]:<15} seed_offset={c["seed_offset"]:<3d} dist={c["distance_negative_class"]:g}')
if len(TABEBM_CONFIGS) > 3:
    print(f'  ... (총 {len(TABEBM_CONFIGS)}개)')

## 4. 병렬화 + 생성 설정

In [ ]:
GPUS            = [0, 1, 2, 3]
# GPUS          = [0, 1]
# GPUS          = [0]

PROCS_PER_GPU   = 7
# PROCS_PER_GPU = 2

# Storage: 500 per class (abundance). 04 evaluator slices: n_syn_total 500 → binary 250/class.
N_SYN_PER_CLASS = 500
# N_SYN_PER_CLASS = 250
# N_SYN_PER_CLASS = 10000          # N_SYN sweep 용

SKIP_EXISTING   = True

N_GPU = len(GPUS)
MAX_WORKERS = N_GPU * PROCS_PER_GPU
print(f'GPUs={GPUS}, workers={MAX_WORKERS}, N_syn_per_class={N_SYN_PER_CLASS}')
print(f'SKIP_EXISTING={SKIP_EXISTING}')


## 5. 샘플 생성 (병렬)

K × n_splits × classes 개 task. 각 task 는 `TabEBM.generate(X_tr[y==c], y==c, seed=split_seed+k)` 호출.

In [ ]:
from fair_eval_worker import run_one_sgld_task
from ensemble_ebm import apply_preprocessor, split_preprocessor_from_npz

data = np.load(EVAL_DIR / 'data.npz')
X_all_raw, y_all = data['X_all'], data['y_all']
sp = np.load(EVAL_DIR / 'splits.npz')
splits = [(sp[f'tr_{i}'], sp[f'val_{i}']) for i in range(config['n_splits'])]

# Paper B.3: per-split preprocessor 적용
X_all_per_split = [
    apply_preprocessor(X_all_raw, split_preprocessor_from_npz(sp, i))
    for i in range(config['n_splits'])
]

ens_dir = str(EVAL_DIR / 'ensembles')
samples_dir = EVAL_DIR / 'samples'
samples_dir.mkdir(exist_ok=True)

# tabebm_sample_config.json merge (기존 유지 + 신규 추가)
_tc_path = EVAL_DIR / 'tabebm_sample_config.json'
if _tc_path.exists():
    _existing = json.loads(_tc_path.read_text())
    _existing_names = {c['name'] for c in _existing.get('tabebm_configs', [])}
else:
    _existing = {'tabebm_configs': []}
    _existing_names = set()
_new_configs = [c for c in TABEBM_CONFIGS if c['name'] not in _existing_names]
_merged = _existing.get('tabebm_configs', []) + _new_configs
_tc_path.write_text(json.dumps({
    'tabebm_configs': _merged,
    'n_syn_per_class': N_SYN_PER_CLASS,
}, indent=2))
print(f'tabebm_sample_config.json: 기존 {len(_existing_names)}개 + 신규 {len(_new_configs)}개 = {len(_merged)}개')

# task 구성
tasks = []; skipped = 0; task_id = 0
for split_i, (tr, _val) in enumerate(splits):
    X_all_s = X_all_per_split[split_i]
    for cfg in TABEBM_CONFIGS:
        for c in classes:
            out_path = samples_dir / f'split_{split_i}' / cfg['name'] / f'c{c}.npy'
            if SKIP_EXISTING and out_path.exists():
                skipped += 1; task_id += 1; continue
            gpu = GPUS[task_id % N_GPU]
            tasks.append(('single', split_i, c, cfg,
                          tr, X_all_s, y_all, N_SYN_PER_CLASS,
                          config['seed'], gpu, ens_dir))
            task_id += 1

n_total = len(tasks)
print(f'{n_total} tasks (skipped {skipped} existing), {MAX_WORKERS} workers')
if n_total == 0:
    print('모든 샘플 이미 존재 — 생성 건너뜀')
else:
    print()
    t0 = time.time(); done = 0
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(run_one_sgld_task, t): t for t in tasks}
        for f in as_completed(futs):
            r = f.result()
            out = samples_dir / f'split_{r["split_i"]}' / r['cfg_name']
            out.mkdir(parents=True, exist_ok=True)
            np.save(out / f'c{r["class_c"]}.npy', r['samples'])
            done += 1
            if done % 10 == 0 or done == n_total:
                elapsed = time.time() - t0
                eta = elapsed / done * (n_total - done)
                print(f'  [{done:>3d}/{n_total}]  '
                      f'elapsed {_fmt(elapsed)}  ETA {_fmt(eta)}', flush=True)
    print(f'\nDone -- {_fmt(time.time()-t0)}')


## 6. 검증

In [ ]:
expected = [c['name'] for c in TABEBM_CONFIGS]
ok = 0; fail = 0
for i in range(config['n_splits']):
    missing = []
    for cn in expected:
        for c in classes:
            if (samples_dir / f'split_{i}' / cn / f'c{c}.npy').exists():
                ok += 1
            else:
                missing.append(f'{cn}/c{c}'); fail += 1
    status = 'OK' if not missing else f'MISSING {missing[:3]}'
    print(f'  split_{i}: {status}')

print(f'\nTotal: {ok} OK, {fail} missing')
print(f'EVAL_DIR = {EVAL_DIR}  --> 04 에 입력')

## 7. (선택) Corner 일치 확인

내 TabEBM seed=split_seed+k 이 만든 corner 가 실제로 01 ensemble member k 의 corner 와 set-level 일치하는지 sanity check.

In [ ]:
from tabebm.TabEBM import TabEBM

def to_set(arr):
    return frozenset(tuple(row) for row in arr)

SPLIT_CHECK = 0
split_seed = config['seed'] + SPLIT_CHECK * 100
all_match = True

print(f'split {SPLIT_CHECK} (split_seed={split_seed}) — 01 member k 의 corner vs TabEBM(seed=split_seed+k)')
print('=' * 70)
for k in range(min(K, 3)):
    # 01 member k class 0 corner 로드
    neg_path = EVAL_DIR / 'ensembles' / f'split_{SPLIT_CHECK}' / 'c0' / f'ebm_{k}' / 'negatives.npz'
    if not neg_path.exists():
        print(f'  [skip] 01 member {k} negatives.npz 없음')
        continue
    corner_01 = np.load(neg_path)['X_neg']

    # TabEBM.add_surrogate_negative_samples 가 같은 seed 에서 만드는 corner
    class_data = np.load(EVAL_DIR / 'ensembles' / f'split_{SPLIT_CHECK}' / 'c0' / 'class_data.npz')
    X_c = class_data['X_class']
    np.random.seed(split_seed + k)   # TabEBM.generate 내부의 seed_everything 과 같은 numpy 상태
    X_ebm_te, y_ebm_te = TabEBM.add_surrogate_negative_samples(
        X_c, distance_negative_class=DIST_NEG)
    corner_te = X_ebm_te[y_ebm_te == 1]

    match = to_set(corner_01) == to_set(corner_te)
    all_match &= match
    print(f'  member {k} (seed={split_seed+k}): 01 corner vs TabEBM corner set-equal = {match}')

if all_match:
    print('\n✅ 모든 확인된 member 에서 corner 일치 — 01 ensemble 과 완전 공정 비교 가능.')
else:
    print('\n⚠ 불일치 — ensemble_methods.py 의 RandomState 수정 확인 필요.')